<a href="https://colab.research.google.com/github/Inkognitochek/analyze-telegram-message-history/blob/main/analyze_message_history.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install telethon openai-whisper
!apt-get install ffmpeg -y



In [ ]:
import os
import csv
import whisper
import nest_asyncio
import asyncio
from datetime import datetime
from google.colab import userdata
from telethon import TelegramClient
from telethon.errors import RPCError

nest_asyncio.apply()

# ================= SAFE SETTINGS =================
API_ID = int(userdata.get('TG_API_ID')) # paste your API_ID
API_HASH = userdata.get('TG_API_HASH')  # paste your API_HASH
CHAT_NAME = '@KOLTETSKI'                # choose username you need

START_FROM_DATE = '2026-09-24 14:00:00' # enter your date
# =================================================

clean_date_str = START_FROM_DATE.strip()
offset_date = datetime.strptime(clean_date_str, '%Y-%m-%d %H:%M:%S')

print("Шаг 1: Загружаем модель Whisper (Small)...")
model = whisper.load_model("small")

client = TelegramClient('my_telegram_session', API_ID, API_HASH, timeout=90)

async def main():
    print(f"Шаг 3: Подключаемся к {CHAT_NAME} и собираем историю с {clean_date_str}...")

    file_name = 'history_koltetski.csv'
    file_exists = os.path.exists(file_name)

    with open(file_name, 'a', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)

        if not file_exists or os.stat(file_name).st_size == 0:
            writer.writerow(['Дата', 'Кто отправил (ID)', 'Текст сообщения', 'Расшифровка голосового'])

        async for message in client.iter_messages(CHAT_NAME, offset_date=offset_date):
            voice_text = ""
            await asyncio.sleep(0.4) # anti-flood tg

            if message.voice:
                for attempt in range(1, 4):
                    try:
                        voice_path = await message.download_media(file='temp_voices/')
                        if voice_path and os.path.exists(voice_path):
                            result = model.transcribe(
                                voice_path,
                                language='ru',
                                temperature=0.0,
                                no_speech_threshold=0.6
                            )
                            voice_text = result['text']
                            os.remove(voice_path)
                        break
                    except (RPCError, Exception) as e:
                        print(f"\n Telegram притормаживает на аудио от {message.date}. Попытка {attempt}/3...")
                        await asyncio.sleep(7)
                        voice_text = f"[Голосовое пропущено из-за лимитов Telegram: {e}]"

            writer.writerow([
                message.date,
                message.sender_id,
                message.text or '',
                voice_text
            ])

    print(f"\n ГОТОВО! Все данные сохранены в файл {file_name}")

async def run():
    await client.start()
    await main()

asyncio.get_event_loop().run_until_complete(run())